# Load Data

In [ ]:
# 2D Motion Compensation using Displacement Tracking
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from scipy.ndimage import binary_fill_holes, binary_dilation, binary_erosion
import os

bmode_path = '/Users/samantha/Desktop/ultrasound lab stuff/raw_ctdna/p16/wk12/combined_nifti/p16_wk12_Fund_RAW.nii'
ceus_path = '/Users/samantha/Desktop/ultrasound lab stuff/raw_ctdna/p16/wk12/combined_nifti/p16_wk12_CHI_normalized_GUI.nii'
small_mc_roi_path = '/Users/samantha/Desktop/ultrasound lab stuff/raw_ctdna/p16/wk12/p16_wk12_small_mc_roi.nii.gz'
large_roi_path = '/Users/samantha/Desktop/ultrasound lab stuff/raw_ctdna/p16/wk12/p16_wk12_static_roi_139.nii.gz'

bmode_data = nib.load(bmode_path).get_fdata()
ceus_data = nib.load(ceus_path).get_fdata()
small_mc_roi_seg = nib.load(small_mc_roi_path).get_fdata()
large_roi_seg = nib.load(large_roi_path).get_fdata()

viz_data = ceus_data  # Use CEUS for visualization

print(f"B-mode: {bmode_data.shape}")
print(f"CEUS: {ceus_data.shape}")
print(f"Small MC ROI: {small_mc_roi_seg.shape}")
print(f"Large ROI: {large_roi_seg.shape}")

# PROCESS COMBINED SIDE-BY-SIDE ROIS (if needed)

In [ ]:
def process_combined_roi(roi_mask, target_shape):
    """Split side-by-side mask and transpose to match target shape."""
    if roi_mask.shape[0] == 1048:
        mid = 524
        left_half = roi_mask[:mid, :, :]
        right_half = roi_mask[mid:, :, :]
        roi_mask = left_half if np.sum(left_half > 0) > np.sum(right_half > 0) else right_half
        if roi_mask.shape != target_shape:
            roi_mask = roi_mask.transpose(1, 0, 2)
    return roi_mask

target_shape = bmode_data.shape
small_mc_roi_seg = process_combined_roi(small_mc_roi_seg, target_shape)
large_roi_seg = process_combined_roi(large_roi_seg, target_shape)

if viz_data.shape[0] == 1048:
    viz_data = process_combined_roi(viz_data, target_shape)

n_frames = bmode_data.shape[2]
print(f"Processed shapes: {small_mc_roi_seg.shape}, {large_roi_seg.shape}")
print(f"Frames: {n_frames}")

# TRACK MOTION FROM SMALL MC ROI

In [ ]:
reference_frame = 139

# Track top-left corner position across frames
top_left_positions = np.zeros((n_frames, 2))
frames_with_roi = []

for t in range(n_frames):
    mask_t = small_mc_roi_seg[:, :, t] > 0
    if np.sum(mask_t) == 0:
        top_left_positions[t] = [-1, -1]
        continue
    frames_with_roi.append(t)
    y_indices, x_indices = np.where(mask_t)
    top_left_positions[t] = [y_indices.min(), x_indices.min()]

# Interpolate missing frames
for t in range(n_frames):
    if top_left_positions[t, 0] == -1:
        valid_before = [f for f in frames_with_roi if f < t]
        valid_after = [f for f in frames_with_roi if f > t]
        if valid_before and valid_after:
            t_before, t_after = max(valid_before), min(valid_after)
            weight = (t - t_before) / (t_after - t_before)
            top_left_positions[t] = (1 - weight) * top_left_positions[t_before] + weight * top_left_positions[t_after]
        elif valid_before:
            top_left_positions[t] = top_left_positions[max(valid_before)]
        elif valid_after:
            top_left_positions[t] = top_left_positions[min(valid_after)]

# Calculate translation vectors relative to reference frame
ref_top_left = top_left_positions[reference_frame]
translation_vectors = np.zeros((n_frames, 3))
for t in range(n_frames):
    dy = top_left_positions[t, 0] - ref_top_left[0]
    dx = top_left_positions[t, 1] - ref_top_left[1]
    translation_vectors[t] = [dx, dy, t]

print(f"Reference frame: {reference_frame}")
print(f"Translation range: X=[{translation_vectors[:, 0].min():.1f}, {translation_vectors[:, 0].max():.1f}], Y=[{translation_vectors[:, 1].min():.1f}, {translation_vectors[:, 1].max():.1f}]")

# =============================================================================
# FILL LARGE ROI IF NEEDED (contour -> solid)
# =============================================================================
test_frame = min(250, n_frames - 1)
roi_test = large_roi_seg[:, :, test_frame]
if np.sum(roi_test > 0) > 0:
    y_coords, x_coords = np.where(roi_test > 0)
    bbox_area = (y_coords.max() - y_coords.min() + 1) * (x_coords.max() - x_coords.min() + 1)
    fill_percentage = (np.sum(roi_test > 0) / bbox_area) * 100
    
    if fill_percentage < 30:
        print("Filling contour ROI...")
        large_roi_seg_filled = np.zeros_like(large_roi_seg)
        for t in range(n_frames):
            roi_frame = large_roi_seg[:, :, t]
            if np.sum(roi_frame > 0) > 0:
                dilated = binary_dilation(roi_frame > 0, iterations=2)
                filled = binary_fill_holes(dilated)
                large_roi_seg_filled[:, :, t] = binary_erosion(filled, iterations=2).astype(np.uint8)
        large_roi_seg = large_roi_seg_filled

# =============================================================================
# APPLY MOTION COMPENSATION TO LARGE ROI
# =============================================================================
large_roi_mc = np.zeros_like(large_roi_seg)

for t in range(n_frames):
    shift_y = int(round(translation_vectors[t, 1]))
    shift_x = int(round(translation_vectors[t, 0]))
    
    shifted = large_roi_seg[:, :, t].copy()
    if shift_y != 0:
        shifted = np.roll(shifted, shift_y, axis=0)
        if shift_y > 0:
            shifted[:shift_y, :] = 0
        else:
            shifted[shift_y:, :] = 0
    if shift_x != 0:
        shifted = np.roll(shifted, shift_x, axis=1)
        if shift_x > 0:
            shifted[:, :shift_x] = 0
        else:
            shifted[:, shift_x:] = 0
    large_roi_mc[:, :, t] = shifted

print(f"Motion compensation complete. ROI pixels: {np.sum(large_roi_mc > 0)}")

# INTERACTIVE VISUALIZATION

In [ ]:
fig_compare, axes_compare = plt.subplots(1, 3, figsize=(18, 6))
plt.close(fig_compare)

def visualize_comparison(frame_idx):
    for ax in axes_compare:
        ax.clear()
    
    axes_compare[0].imshow(viz_data[:, :, frame_idx], cmap='gray')
    axes_compare[0].contour(small_mc_roi_seg[:, :, frame_idx], colors='green', linewidths=2)
    axes_compare[0].set_title('Small MC ROI (tracking)')
    axes_compare[0].axis('off')
    
    axes_compare[1].imshow(viz_data[:, :, frame_idx], cmap='gray')
    axes_compare[1].contour(large_roi_seg[:, :, frame_idx], colors='red', linewidths=2)
    axes_compare[1].set_title('Large ROI (static)')
    axes_compare[1].axis('off')
    
    axes_compare[2].imshow(viz_data[:, :, frame_idx], cmap='gray')
    axes_compare[2].contour(large_roi_mc[:, :, frame_idx], colors='cyan', linewidths=2)
    axes_compare[2].set_title(f'Motion Corrected (dx={translation_vectors[frame_idx, 0]:.1f}, dy={translation_vectors[frame_idx, 1]:.1f})')
    axes_compare[2].axis('off')
    
    fig_compare.suptitle(f'Frame {frame_idx}')
    fig_compare.tight_layout()

frame_slider = widgets.IntSlider(value=reference_frame, min=0, max=n_frames-1, step=1, description='Frame:', continuous_update=False)
output_widget = widgets.Output()

def on_frame_change(change):
    with output_widget:
        clear_output(wait=True)
        visualize_comparison(change['new'])
        display(fig_compare)

frame_slider.observe(on_frame_change, names='value')
display(frame_slider, output_widget)
frame_slider.value = reference_frame

# SAVE RESULTS

In [ ]:
roi_dir = os.path.dirname(large_roi_path)
roi_filename = os.path.basename(large_roi_path)
roi_name_no_ext = roi_filename.replace('.nii.gz', '').replace('.nii', '')

output_path = os.path.join(roi_dir, f"{roi_name_no_ext}_displacement_mc.nii.gz")
nib.save(nib.Nifti1Image(large_roi_mc.astype(np.uint8), np.eye(4)), output_path)
print(f"Saved: {output_path}")

vectors_path = os.path.join(roi_dir, f"{roi_name_no_ext}_translation_vectors.npy")
np.save(vectors_path, translation_vectors)
print(f"Saved: {vectors_path}")

# EXPORT VIDEO

In [ ]:
from matplotlib.animation import FFMpegWriter

def export_mc_video(output_path=None, fps=30, frame_range=None):
    """Export motion corrected ROI overlay as video."""
    if output_path is None:
        output_path = os.path.join(roi_dir, f"{roi_name_no_ext}_mc_video.mp4")
    
    start_frame = frame_range[0] if frame_range else 0
    end_frame = frame_range[1] if frame_range else n_frames
    total_frames = end_frame - start_frame
    
    print(f"Exporting: {output_path}")
    print(f"Frames {start_frame}-{end_frame-1} ({total_frames} frames) @ {fps} fps")
    
    fig, ax = plt.subplots(figsize=(8, 6))
    writer = FFMpegWriter(fps=fps)
    
    with writer.saving(fig, output_path, dpi=100):
        for i, t in enumerate(range(start_frame, end_frame)):
            ax.clear()
            ax.imshow(viz_data[:, :, t], cmap='gray')
            ax.contour(large_roi_mc[:, :, t], colors='cyan', linewidths=1.5)
            ax.set_title(f'Frame {t} | dx={translation_vectors[t, 0]:.1f}, dy={translation_vectors[t, 1]:.1f}')
            ax.axis('off')
            writer.grab_frame()
            
            if i % 50 == 0 or i == total_frames - 1:
                clear_output(wait=True)
                display(fig)
                print(f"Progress: {i+1}/{total_frames} ({100*(i+1)/total_frames:.0f}%)")
    
    plt.close(fig)
    clear_output(wait=True)
    print(f"Saved: {output_path}")

export_mc_video()